# E-commerce: Data Exploration with DuckDB

In this notebook we load 5 CSV files into tables using DuckDB 
and explore them with SQL designing the data warehouse.

**What we will do**

1. Load the CSV files into DuckDB
2. Understand each table: structure, grain, keys
3. Check relationships (foreign keys)
4. Find the data quality issues
5. Preview the star schema
6. Run OLAP-style queries (roll-up, slice, dice, drill-down)

**Tables**: 'customers', 'orders', 'order_items', 'products', 'sellers'

In [6]:
from pathlib import Path

import pandas as pd
import duckdb

In [7]:
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
WAREHOUSE_DIR = DATA_DIR / "warehouse"

csv = RAW_DIR / "olist_orders_dataset.csv"

In [9]:
print(duckdb.sql(f"SELECT count(*) from '{csv}'"))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        99441 │
└──────────────┘



## DuckDB lets us query CSV data with SQL. The databse file persists locally so later notebooks can reuse loaded raw tables.

In [14]:
DB_PATH = WAREHOUSE_DIR / "ecommerce.duckdb"
con = duckdb.connect(str(DB_PATH))

## 1. Define helper function

In [15]:
def q(sql:str) -> pd.DataFrame:
    """ Runs a SQL query in DuckDB and returns a DataFrame."""
    return con.sql(sql).df()

## 2. Load the source tables into a **raw schema**

A schema is a namespace that groups tables. We keep the source copies in **raw** so that we can differentiate it from the cleaned warehouse tables.

In [16]:
RAW_TABLES: dict[str, dict[str, str]] = {
    "customers": {"customer_zip_code_prefix": "VARCHAR"}, # keep leading zeros
    "orders": {},
    "order_items": {},
    "products": {},
    "sellers": {}
}

In [17]:
# Define raw sche,ma
RAW_SCHEMA = "raw"

In [18]:
def csv_path(table:str) -> Path:
    return RAW_DIR /f"olist_{table}_dataset.csv"

In [20]:
# Load raw data into tables
def load_raw_table(con: duckdb.DuckDBPyConnection, table: str, column_types: dict[str, str]) -> int:
    """Create a raw.<table> from its csv and return the raw count"""
    target = f'{RAW_SCHEMA}."{table}"'
    source = con.read_csv(str(csv_path(table)), header=True, dtype=column_types or None)
    con.execute(f"DROP TABLE IF EXISTS {target}")
    source.to_table(target)
    return con.execute(f"SELECT count(*) FROM {target}").fetchone()[0]

In [21]:
# Create the schema
con.execute(f"CREATE SCHEMA IF NOT EXISTS {RAW_SCHEMA}")

In [22]:
row_counts = {t: load_raw_table(con, t, types) for t, types in RAW_TABLES.items()}

In [24]:
pd.DataFrame(row_counts.items(), columns=["table_name", "row_count"])

,table_name,row_count
0,customers,99441
1,orders,99441
2,order_items,112650
3,products,32951
4,sellers,3095


In [25]:
q("""
    SELECT 
        customer_id, 
        customer_zip_code_prefix 
    FROM raw.customers LIMIT 5
""")

,customer_id,customer_zip_code_prefix
0,06b8999e2fba1a1fbc88172c00ba8bc7,14409
1,18955e83d337fd6b2def6b18a428ac77,09790
2,4e7b3e00288586ebd08712fdd0374a03,01151
3,b2b6027bc5c5109e529d4dc6358b12c3,08775
4,4f2d8ab171c80ec8364f7c12e35b23ad,13056


In [26]:
for t in RAW_TABLES:
    print(f"------ {t} -----")
    display(q(f"DESCRIBE raw.{t}")[["column_name", "column_type", "null"]])

------ customers -----


,column_name,column_type,null
0,customer_id,VARCHAR,YES
1,customer_unique_id,VARCHAR,YES
2,customer_zip_code_prefix,VARCHAR,YES
3,customer_city,VARCHAR,YES
4,customer_state,VARCHAR,YES


------ orders -----


,column_name,column_type,null
0,order_id,VARCHAR,YES
1,customer_id,VARCHAR,YES
2,order_status,VARCHAR,YES
3,order_purchase_timestamp,TIMESTAMP,YES
4,order_approved_at,TIMESTAMP,YES
5,order_delivered_carrier_date,TIMESTAMP,YES
6,order_delivered_customer_date,TIMESTAMP,YES
7,order_estimated_delivery_date,TIMESTAMP,YES


------ order_items -----


,column_name,column_type,null
0,order_id,VARCHAR,YES
1,order_item_id,BIGINT,YES
2,product_id,VARCHAR,YES
3,seller_id,VARCHAR,YES
4,shipping_limit_date,TIMESTAMP,YES
5,price,DOUBLE,YES
6,freight_value,DOUBLE,YES


------ products -----


,column_name,column_type,null
0,product_id,VARCHAR,YES
1,product_category_name,VARCHAR,YES
2,product_name_lenght,BIGINT,YES
3,product_description_lenght,BIGINT,YES
4,product_photos_qty,BIGINT,YES
5,product_weight_g,BIGINT,YES
6,product_length_cm,BIGINT,YES
7,product_height_cm,BIGINT,YES
8,product_width_cm,BIGINT,YES


------ sellers -----


,column_name,column_type,null
0,seller_id,VARCHAR,YES
1,seller_zip_code_prefix,VARCHAR,YES
2,seller_city,VARCHAR,YES
3,seller_state,VARCHAR,YES


## Data Quality - write SQL scripts to assess quality issues and write notes about the impact
#### Orders with no items
#### Order status and class imbalance
#### Missing and inconsistent timestamps
## ERD Diagram
#### Design ERD for the 5 tables
#### Relationship, cardinality, primary-key, foreign-key, parent-child, ...
##### cardinality - breakdown the many-many relations with another table - eg breakdown order-products with orderitems